# 2/3 - EDA Fitur Gambar, Univariate Outlier, ANOVA vs IG/MI, dan Hybrid PCA

Notebook ini membaca `dataset_final_manifest.csv`, mengekstraksi fitur dari dataset final yang sama, lalu menjalankan:
1. analisis univariate dan outlier per fitur awal;
2. korelasi dan reduksi redundansi;
3. ANOVA F-score sebagai baseline statistik berbasis perbedaan rerata;
4. Information Gain/Mutual Information sebagai pembanding dependency nonlinear;
5. penentuan `Core Stable Features` dari irisan Top ANOVA dan Top IG/MI;
6. peringkasan fitur residual melalui Hybrid PCA;
7. penyimpanan final feature set untuk notebook 3.

Output penting untuk notebook 3:
- `final_feature_set_for_multivariate_outlier.csv`
- `hybrid_pca_final_feature_matrix.csv`


**Update:** Notebook ini sekarang secara eksplisit mengekspor `final_feature_set_for_multivariate_outlier.csv` sebagai input wajib untuk notebook 3.

# EDA Fitur Gambar - Versi Konsisten + ANOVA/IG Core Features + Hybrid PCA

Perbaikan utama:
1. Input fitur dikunci ke `dataset_final_manifest.csv` yang dibuat oleh notebook `eda_datasetawal_fixed.ipynb`.
2. Ekstraksi fitur tidak lagi bergantung pada scanning folder saja, sehingga jumlah data konsisten dengan dataset final setelah deduplikasi tahap kedua.
3. CSV lama yang tidak cocok dengan manifest akan dibersihkan dari path stale.
4. Ranking fitur univariate tidak hanya memakai ANOVA F-score, tetapi juga Information Gain/Mutual Information.
5. Karena distribusi kelas dapat tidak seimbang, perbandingan ANOVA vs IG dihitung melalui balanced bootstrap per kelas.


In [ ]:
import os
import re
import warnings
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import cv2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.stats import skew, kurtosis
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.measure import shannon_entropy
from skimage import measure, filters, morphology

cv2.setNumThreads(1)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 120

CLEAN_DATASET_DIR = Path("dataset_cleaned_preprocessed")
FINAL_MANIFEST_CSV = Path("dataset_final_manifest.csv")
FEATURE_CSV = Path("features_extracted_clean_final.csv")

# False menjaga resume dari CSV parsial; set True hanya jika ingin hitung ulang dari nol.
FORCE_RECOMPUTE_FEATURES = False

def get_numeric_feature_columns(df, exclude=("original_path", "class", "is_outlier")):
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def sanitize_filename(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text).strip().lower()).strip("_")


In [ ]:
import numpy as np
from numba import jit

@jit(nopython=True)
def _manual_linear_regression(x, y):
    """Implementasi manual Least Squares untuk menggantikan np.polyfit."""
    n = len(x)
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xx = np.sum(x * x)
    sum_xy = np.sum(x * y)
    
    # Rumus gradien (slope) m = (n*sum(xy) - sum(x)*sum(y)) / (n*sum(xx) - sum(x)^2)
    denominator = (n * sum_xx - sum_x**2)
    if denominator == 0:
        return 0.0
    slope = (n * sum_xy - sum_x * sum_y) / denominator
    return slope

@jit(nopython=True)
def calculate_fractal_dimension(img_array):
    """Differential Box Counting (DBC) yang kompatibel dengan Numba."""
    img_array = img_array.astype(np.float64)
    h, w = img_array.shape
    max_box_size = min(h, w) // 2
    
    # Skala box (power of 2)
    scales = []
    val = 2
    while val <= max_box_size:
        scales.append(val)
        val *= 2
    
    scales_arr = np.array(scales, dtype=np.float64)
    n_boxes = np.zeros(len(scales_arr))
    
    for idx in range(len(scales_arr)):
        s = int(scales_arr[idx])
        count = 0.0
        for i in range(0, h - s, s):
            for j in range(0, w - s, s):
                box = img_array[i:i+s, j:j+s]
                # DBC Logic
                count += np.ceil((np.max(box) - np.min(box)) / s) + 1
        n_boxes[idx] = count
    
    # Log-Log Transformation
    log_inv_scales = np.log(1.0 / scales_arr)
    log_n_boxes = np.log(n_boxes)
    
    # Gunakan regresi manual sebagai pengganti polyfit
    return _manual_linear_regression(log_inv_scales, log_n_boxes) 

# Referensi:
# [1] N. Sarkar and B. B. Chaudhuri, "An efficient differential box-counting approach to compute fractal dimension of image," IEEE Trans. Syst., Man, Cybern., vol. 24, no. 1, 1994.


In [ ]:
@jit(nopython=True)
def calculate_lacunarity_numba(img_array, box_size=4):
    """Menghitung lacunarity menggunakan gliding box."""
    h, w = img_array.shape
    img_float = img_array.astype(np.float64)

    masses = []
    for i in range(0, h - box_size + 1, box_size):
        for j in range(0, w - box_size + 1, box_size):
            window = img_float[i:i+box_size, j:j+box_size]
            masses.append(np.sum(window))

    if len(masses) == 0:
        return 0.0

    mass_arr = np.array(masses)
    mean_m = np.mean(mass_arr)
    if mean_m == 0:
        return 0.0

    return np.mean(mass_arr**2) / (mean_m**2)

@jit(nopython=True)
def calculate_bimodal_coeff_numba(img_array):
    """Bimodal coefficient versi aman untuk citra konstan."""
    pix = img_array.ravel().astype(np.float64)
    n = len(pix)
    if n < 4:
        return 0.0

    mu = np.mean(pix)
    std = np.std(pix)
    if std == 0:
        return 0.0

    s3 = 0.0
    s4 = 0.0
    for i in range(n):
        diff = (pix[i] - mu) / std
        s3 += diff**3
        s4 += diff**4

    skew_val = s3 / n
    kurt_val = s4 / n
    if kurt_val == 0:
        return 0.0

    return (skew_val**2 + 1.0) / kurt_val

@jit(nopython=True)
def calculate_mean_gradient_numba(img_array):
    """Menghitung rerata kekuatan edge Sobel-like."""
    h, w = img_array.shape
    if h < 3 or w < 3:
        return 0.0

    grad_sum = 0.0
    count = 0
    for i in range(1, h-1):
        for j in range(1, w-1):
            gx = (img_array[i-1, j+1] + 2*img_array[i, j+1] + img_array[i+1, j+1]) -                  (img_array[i-1, j-1] + 2*img_array[i, j-1] + img_array[i+1, j-1])
            gy = (img_array[i+1, j-1] + 2*img_array[i+1, j] + img_array[i+1, j+1]) -                  (img_array[i-1, j-1] + 2*img_array[i-1, j] + img_array[i-1, j+1])
            grad_sum += np.sqrt(gx**2 + gy**2)
            count += 1

    return grad_sum / count if count > 0 else 0.0


In [ ]:
def process_single_image_all_features(img_path):
    """Ekstraksi fitur EDA untuk satu gambar grayscale/PNG."""
    try:
        with Image.open(img_path) as img:
            img = ImageOps.exif_transpose(img).convert("L")
            img_arr = np.array(img).astype(np.uint8)

        px = img_arr.flatten().astype(np.float64)

        d_val = calculate_fractal_dimension(img_arr)
        lacunarity = calculate_lacunarity_numba(img_arr, box_size=4)

        glcm = graycomatrix(
            img_arr,
            distances=[1],
            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
            levels=256,
            symmetric=True,
            normed=True
        )

        lbp = local_binary_pattern(img_arr, 8, 1, method="uniform")
        hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
        lbp_hist = hist.astype(float) / (hist.sum() + 1e-7)

        blur_score = cv2.Laplacian(img_arr, cv2.CV_64F).var()
        bimodal_coeff = calculate_bimodal_coeff_numba(img_arr)

        mean_px = float(np.mean(px))
        std_px = float(np.std(px))
        cv_score = std_px / abs(mean_px) if abs(mean_px) > 1e-12 else 0.0

        f_shift = np.fft.fftshift(np.fft.fft2(img_arr))
        psd = np.abs(f_shift)**2
        psd_norm = psd / (np.sum(psd) + 1e-12)
        spec_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

        # Morfologi: jika Otsu gagal pada citra konstan, gunakan fallback median.
        try:
            thresh = filters.threshold_otsu(img_arr)
        except Exception:
            thresh = np.median(img_arr)

        binary = img_arr < thresh
        binary = morphology.remove_small_objects(binary, min_size=50)
        labels = measure.label(binary)
        props = measure.regionprops(labels)

        solidity = 0.0
        aspect_ratio = 1.0
        if props:
            main_obj = max(props, key=lambda x: x.area)
            solidity = float(main_obj.solidity)
            aspect_ratio = float(main_obj.major_axis_length / (main_obj.minor_axis_length + 1e-6))

        mean_grad = calculate_mean_gradient_numba(img_arr)

        res = {
            "original_path": str(Path(img_path)),
            "class": Path(img_path).parent.name,
            "fractal_dimension": d_val,
            "lacunarity": lacunarity,
            "entropy": shannon_entropy(img_arr),
            "mean_intensity": mean_px,
            "std_intensity": std_px,
            "skewness": skew(px),
            "kurtosis": kurtosis(px),
            "cv_intensity": cv_score,
            "bimodal_coeff": bimodal_coeff,
            "glcm_contrast": float(np.mean(graycoprops(glcm, "contrast"))),
            "glcm_homogeneity": float(np.mean(graycoprops(glcm, "homogeneity"))),
            "glcm_energy": float(np.mean(graycoprops(glcm, "energy"))),
            "glcm_correlation": float(np.mean(graycoprops(glcm, "correlation"))),
            "solidity": solidity,
            "aspect_ratio": aspect_ratio,
            "blur_score": blur_score,
            "spectral_entropy": spec_entropy,
            "mean_gradient": mean_grad,
        }

        for i, val in enumerate(lbp_hist):
            res[f"lbp_bin_{i}"] = float(val)

        # Normalisasi finite values agar analisis statistik tidak rusak.
        for k, v in list(res.items()):
            if isinstance(v, (int, float, np.floating)) and not np.isfinite(v):
                res[k] = np.nan

        return res

    except Exception as e:
        return {"original_path": str(img_path), "error": str(e)}


In [ ]:
def load_final_manifest(manifest_csv=FINAL_MANIFEST_CSV, fallback_dir=CLEAN_DATASET_DIR):
    """Memuat daftar gambar final dari manifest resmi.

    Manifest dibuat setelah deduplikasi tahap kedua pada notebook dataset awal.
    Jika manifest tidak ditemukan, fungsi fallback ke scanning folder, tetapi kondisi ini
    sebaiknya hanya dipakai untuk debugging, bukan untuk hasil final skripsi.
    """
    manifest_csv = Path(manifest_csv)
    fallback_dir = Path(fallback_dir)

    if manifest_csv.exists():
        df_manifest = pd.read_csv(manifest_csv)
        required = {"path", "class", "filename"}
        missing = required - set(df_manifest.columns)
        if missing:
            raise ValueError(f"Manifest {manifest_csv} tidak memiliki kolom wajib: {missing}")

        df_manifest["path"] = df_manifest["path"].astype(str)
        df_manifest["path_obj"] = df_manifest["path"].apply(lambda p: Path(p))
        df_manifest["exists"] = df_manifest["path_obj"].apply(lambda p: p.exists())

        if not df_manifest["exists"].all():
            missing_files = df_manifest.loc[~df_manifest["exists"], "path"].head(10).tolist()
            raise FileNotFoundError(
                "Ada file pada dataset_final_manifest.csv yang tidak ditemukan. "
                f"Contoh path hilang: {missing_files}"
            )

        df_manifest = df_manifest.sort_values(["class", "filename", "path"]).reset_index(drop=True)
        print(f"✅ Manifest final dimuat: {manifest_csv} | {len(df_manifest)} gambar")
        display(df_manifest["class"].value_counts().rename_axis("class").reset_index(name="count"))
        return df_manifest

    # Fallback hanya untuk debugging.
    if not fallback_dir.exists():
        raise FileNotFoundError(
            f"Manifest tidak ditemukan: {manifest_csv.resolve()} dan folder fallback tidak ada: {fallback_dir.resolve()}"
        )

    rows = []
    for p in sorted(fallback_dir.rglob("*.png")):
        rows.append({"path": str(p), "class": p.parent.name, "filename": p.name, "path_obj": p, "exists": True})

    df_manifest = pd.DataFrame(rows)
    if df_manifest.empty:
        raise ValueError(f"Tidak ada PNG ditemukan pada folder fallback: {fallback_dir.resolve()}")

    print("⚠️ Manifest final tidak ditemukan. Notebook memakai fallback scanning folder.")
    print("   Untuk hasil final skripsi, jalankan eda_datasetawal_fixed.ipynb hingga membuat dataset_final_manifest.csv.")
    display(df_manifest["class"].value_counts().rename_axis("class").reset_index(name="count"))
    return df_manifest


df_manifest = load_final_manifest()
FINAL_IMAGE_PATHS = [Path(p) for p in df_manifest["path"].tolist()]
FINAL_PATH_SET = set(map(str, FINAL_IMAGE_PATHS))


In [ ]:
import json
import psutil
from joblib import Parallel, delayed
from tqdm.auto import tqdm

FEATURE_EXTRACTION_BACKEND = os.getenv("EDA_FEATURE_BACKEND", "threading").strip().lower()
FEATURE_EXTRACTION_MAX_WORKERS = int(os.getenv("EDA_FEATURE_MAX_WORKERS", "6"))
FEATURE_EXTRACTION_RAM_GB_PER_WORKER = float(os.getenv("EDA_FEATURE_RAM_GB_PER_WORKER", "1.5"))
FEATURE_EXTRACTION_BATCH_SIZE = int(os.getenv("EDA_FEATURE_BATCH_SIZE", "128"))
FEATURE_EXTRACTION_MIN_AVAILABLE_RAM_GB = float(os.getenv("EDA_FEATURE_MIN_AVAILABLE_RAM_GB", "2.0"))
FEATURE_EXTRACTION_TARGET_IDLE_CPU_PERCENT = float(os.getenv("EDA_FEATURE_TARGET_IDLE_CPU_PERCENT", "25"))
FEATURE_EXTRACTION_MAX_CPU_PERCENT = float(os.getenv("EDA_FEATURE_MAX_CPU_PERCENT", "85"))
FEATURE_RUNTIME_CONFIG_JSON = Path("eda_feature_runtime_config.json")

def load_feature_runtime_config():
    defaults = {
        "backend": FEATURE_EXTRACTION_BACKEND,
        "max_workers": FEATURE_EXTRACTION_MAX_WORKERS,
        "ram_gb_per_worker": FEATURE_EXTRACTION_RAM_GB_PER_WORKER,
        "batch_size": FEATURE_EXTRACTION_BATCH_SIZE,
        "min_available_ram_gb": FEATURE_EXTRACTION_MIN_AVAILABLE_RAM_GB,
        "target_idle_cpu_percent": FEATURE_EXTRACTION_TARGET_IDLE_CPU_PERCENT,
        "max_cpu_percent": FEATURE_EXTRACTION_MAX_CPU_PERCENT,
    }
    if not FEATURE_RUNTIME_CONFIG_JSON.exists():
        FEATURE_RUNTIME_CONFIG_JSON.write_text(json.dumps(defaults, indent=2), encoding="utf-8")
        return defaults

    user_config = json.loads(FEATURE_RUNTIME_CONFIG_JSON.read_text(encoding="utf-8"))
    defaults.update({k: v for k, v in user_config.items() if k in defaults})
    defaults["backend"] = str(defaults["backend"]).strip().lower()
    defaults["max_workers"] = max(1, int(defaults["max_workers"]))
    defaults["batch_size"] = max(1, int(defaults["batch_size"]))
    defaults["ram_gb_per_worker"] = max(0.25, float(defaults["ram_gb_per_worker"]))
    defaults["min_available_ram_gb"] = max(0.0, float(defaults["min_available_ram_gb"]))
    defaults["target_idle_cpu_percent"] = min(95.0, max(0.0, float(defaults["target_idle_cpu_percent"])))
    defaults["max_cpu_percent"] = min(100.0, max(1.0, float(defaults["max_cpu_percent"])))
    return defaults

def choose_adaptive_feature_workers(task_count, config):
    cpu_count = os.cpu_count() or 1
    cpu_percent = psutil.cpu_percent(interval=0.25)
    available_ram_gb = psutil.virtual_memory().available / (1024**3)

    usable_cpu_percent = max(0.0, 100.0 - max(cpu_percent, config["target_idle_cpu_percent"]))
    cpu_workers = max(1, int(cpu_count * usable_cpu_percent / 100.0))
    if cpu_percent >= config["max_cpu_percent"] or available_ram_gb < config["min_available_ram_gb"]:
        cpu_workers = 1

    ram_workers = max(1, int(available_ram_gb // config["ram_gb_per_worker"]))
    n_workers = max(1, min(cpu_workers, ram_workers, config["max_workers"], task_count))
    probe = f"cpu={cpu_percent:.0f}% ram={available_ram_gb:.1f}GB"
    return n_workers, probe

def clean_feature_dataframe(df, all_images, current_paths):
    if df.empty:
        return df
    df = df.drop_duplicates("original_path")
    df = df[df["original_path"].astype(str).isin(current_paths)]

    order_map = {str(p): i for i, p in enumerate(all_images)}
    df["_manifest_order"] = df["original_path"].map(order_map)
    df = df.sort_values("_manifest_order").drop(columns="_manifest_order").reset_index(drop=True)

    numeric_cols = get_numeric_feature_columns(df)
    if numeric_cols:
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
        df[numeric_cols] = df[numeric_cols].apply(lambda s: s.fillna(s.median()), axis=0)
    return df

def run_feature_tasks_with_fallback(tasks, n_workers, backend):
    """Jalankan ekstraksi dengan fallback serial jika backend paralel gagal."""
    if n_workers <= 1:
        return [process_single_image_all_features(p) for p in tqdm(tasks, desc="Feature Extraction")]

    try:
        return Parallel(n_jobs=n_workers, backend=backend)(
            delayed(process_single_image_all_features)(p)
            for p in tqdm(tasks, desc=f"Feature Extraction ({backend})")
        )
    except Exception as exc:
        print(f"Parallel backend '{backend}' gagal: {type(exc).__name__}: {exc}")
        print("Fallback ke mode serial agar ekstraksi tetap berjalan dan dapat di-resume dari CSV.")
        return [process_single_image_all_features(p) for p in tqdm(tasks, desc="Feature Extraction (serial fallback)")]

def run_feature_extraction_parallel_dynamic(image_paths, output_csv, force_recompute=False):
    """Ekstraksi fitur dengan resume yang aman terhadap CSV lama/stale.

    Parameter `image_paths` wajib berasal dari dataset_final_manifest.csv agar jumlah data
    konsisten dengan dataset final setelah deduplikasi tahap kedua.
    """
    output_path = Path(output_csv)
    all_images = [Path(p) for p in image_paths]

    if not all_images:
        raise ValueError("Daftar image_paths kosong. Periksa dataset_final_manifest.csv.")

    missing = [str(p) for p in all_images if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Ada {len(missing)} file manifest yang tidak ditemukan. Contoh: {missing[:5]}")

    current_paths = {str(p) for p in all_images}

    initial_config = load_feature_runtime_config()
    cpu_count = os.cpu_count() or 1
    available_ram_gb = psutil.virtual_memory().available / (1024**3)

    print(f"📡 System Probe: {cpu_count} CPU(s) | {available_ram_gb:.1f}GB RAM Available")
    print(f"📁 Manifest images: {len(all_images)}")
    print(f"Runtime config: {FEATURE_RUNTIME_CONFIG_JSON}")
    print(f"Backend awal: {initial_config['backend']} | Max workers: {initial_config['max_workers']} | Batch size: {initial_config['batch_size']}")
    print("-" * 60)

    if force_recompute or not output_path.exists():
        df_old = pd.DataFrame()
        processed_paths = set()
        tasks = all_images
        if force_recompute and output_path.exists():
            print(f"♻️ FORCE_RECOMPUTE=True, CSV lama akan ditimpa: {output_path}")
    else:
        df_old = pd.read_csv(output_path)
        if "original_path" not in df_old.columns:
            raise ValueError(f"CSV lama {output_path} tidak memiliki kolom original_path.")

        # Buang baris stale yang tidak termasuk manifest final.
        before = len(df_old)
        df_old = df_old[df_old["original_path"].astype(str).isin(current_paths)].drop_duplicates("original_path")
        after = len(df_old)
        if before != after:
            print(f"🧹 Membersihkan {before - after} baris stale dari CSV lama.")

        processed_paths = set(df_old["original_path"].astype(str))
        tasks = [p for p in all_images if str(p) not in processed_paths]

    df_work = df_old.copy()
    all_error_data = []
    if tasks:
        total_tasks = len(tasks)
        batch_start = 0
        while batch_start < total_tasks:
            config = load_feature_runtime_config()
            batch_size = config["batch_size"]
            batch = tasks[batch_start:batch_start + batch_size]
            n_workers, probe = choose_adaptive_feature_workers(len(batch), config)
            batch_end = batch_start + len(batch)

            print(
                f"Batch {batch_start + 1}-{batch_end}/{total_tasks} | "
                f"backend={config['backend']} workers={n_workers} | {probe}"
            )
            results = run_feature_tasks_with_fallback(batch, n_workers, config["backend"])
            valid_data = [r for r in results if r is not None and "error" not in r]
            error_data = [r for r in results if r is not None and "error" in r]
            all_error_data.extend(error_data)

            if valid_data:
                df_work = pd.concat([df_work, pd.DataFrame(valid_data)], ignore_index=True)
                df_work = clean_feature_dataframe(df_work, all_images, current_paths)
                df_work.to_csv(output_path, index=False)
                print(f"Checkpoint CSV parsial: {output_path} | {len(df_work)}/{len(all_images)} baris")

            if all_error_data:
                pd.DataFrame(all_error_data).to_csv("feature_extraction_errors.csv", index=False)
                print(f"Ada {len(all_error_data)} error. Detail: feature_extraction_errors.csv")

            batch_start = batch_end
    else:
        print("✅ Semua file pada manifest final sudah ada di CSV.")

    df_final = df_work

    if df_final.empty:
        raise ValueError("Ekstraksi fitur menghasilkan DataFrame kosong.")

    df_final = clean_feature_dataframe(df_final, all_images, current_paths)

    expected_n = len(all_images)
    actual_n = len(df_final)
    if actual_n != expected_n:
        raise ValueError(
            f"Jumlah fitur tidak konsisten dengan manifest: fitur={actual_n}, manifest={expected_n}. "
            "Periksa error extraction atau file gambar yang rusak."
        )

    df_final.to_csv(output_path, index=False)
    print(f"🏁 Selesai. CSV final: {output_path} | {len(df_final)} baris")
    display(df_final["class"].value_counts().rename_axis("class").reset_index(name="count"))

    return df_final


In [ ]:
# --- EKSEKUSI EKSTRAKSI FITUR ---
df_features = run_feature_extraction_parallel_dynamic(
    FINAL_IMAGE_PATHS,
    FEATURE_CSV,
    force_recompute=FORCE_RECOMPUTE_FEATURES
)

numeric_features = get_numeric_feature_columns(df_features)
print(f"Jumlah fitur numerik: {len(numeric_features)}")

# Validasi konsistensi jumlah data dengan manifest final
assert len(df_features) == len(df_manifest), (
    f"Jumlah df_features ({len(df_features)}) tidak sama dengan manifest final ({len(df_manifest)})."
)

class_balance_table = df_features["class"].value_counts().rename_axis("class").reset_index(name="count")
class_balance_table["proportion"] = class_balance_table["count"] / class_balance_table["count"].sum()
display(class_balance_table)

if class_balance_table["count"].nunique() != 1:
    print("⚠️ Distribusi kelas final tidak seimbang. Ranking univariate akan dihitung dengan balanced bootstrap per kelas.")
else:
    print("✅ Distribusi kelas final seimbang.")


In [ ]:
df_features.columns

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
def generate_class_statistics(df, features):
    """Statistik deskriptif mean, std, median, min, max per kelas."""
    stats = df.groupby("class")[features].agg(["mean", "std", "median", "min", "max"])
    stats.columns = ["_".join(col).strip() for col in stats.columns.values]
    return stats

numeric_features = get_numeric_feature_columns(df_features)
df_stats = generate_class_statistics(df_features, numeric_features)

print("\nRingkasan Statistik Deskriptif (5 Fitur Pertama):")
display(df_stats.iloc[:, :5])

df_stats.to_csv("hasil_statistika_deskriptif_per_kelas.csv")


In [ ]:
df_stats.columns.tolist()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_step1_multi_bar(df_stats, features_list, category_name="Tekstur"):
    """
    Plotting multiple bar charts untuk satu kategori fitur.
    Mendukung pilar Validasi Statistika Ketat dengan Error Bars (Std Dev).
    """
    stats = df_stats.reset_index()
    n_features = len(features_list)
    n_cols = 2 if n_features > 1 else 1
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    if n_features > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, feature in enumerate(features_list):
        mean_col = f'{feature}_mean'
        std_col = f'{feature}_std'
        
        # Validasi ketersediaan kolom
        if mean_col not in stats.columns:
            continue
            
        colors = sns.color_palette('viridis', n_colors=len(stats))
        
        # Plotting pada subplot
        bars = axes[i].bar(stats['class'], stats[mean_col], yerr=stats[std_col], 
                           capsize=6, color=colors, alpha=0.7, edgecolor='black')
        
        axes[i].set_title(f'Mean & Std: {feature}', fontsize=12, fontweight='bold')
        axes[i].set_ylabel('Nilai')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Menambahkan label angka di atas setiap bar
        for bar in bars:
            yval = bar.get_height()
            axes[i].text(bar.get_x() + bar.get_width()/2, yval + (stats[std_col].max()*0.15), 
                         f'{yval:.2f}', va='bottom', ha='center', fontsize=8)

    # Menghapus subplot kosong jika jumlah fitur ganjil
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.suptitle(f'Analisis Kategori: {category_name}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Simpan untuk dokumentasi MLOps/Tesis
    file_name = f"step1_barplot_{category_name.lower().replace(' ', '_')}.png"
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    print(f"Plot kategori {category_name} berhasil disimpan sebagai {file_name}")
    plt.show()

# --- EKSEKUSI BERTAHAP PER KATEGORI ---

# 1. Kategori Non-Euclidean (Complexity)
texture_noneuc = ['fractal_dimension', 'lacunarity', 'entropy']
plot_step1_multi_bar(df_stats, texture_noneuc, "Non-Euclidean Complexity")

# 2. Kategori Spatial Texture (GLCM)
texture_glcm = ['glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
plot_step1_multi_bar(df_stats, texture_glcm, "Spatial Texture (GLCM)")

# 3. Kategori Intensitas (Sudah tepat)
intensity_keys = ['mean_intensity', 'std_intensity', 'skewness', 'kurtosis', 'bimodal_coeff', 'cv_intensity']
plot_step1_multi_bar(df_stats, intensity_keys, "Intensity & Histogram Stats")

# 4. Kategori Morphology (Geometry)
morph_keys = ['solidity', 'aspect_ratio']
plot_step1_multi_bar(df_stats, morph_keys, "Object Morphology")

# 5. Kategori Frequency Domain & Quality
quality_keys = ['spectral_entropy', 'blur_score', 'mean_gradient']
plot_step1_multi_bar(df_stats, quality_keys, "Frequency & Image Quality")

# 6. Kategori Micro-Texture (LBP)
lbp_keys = [f'lbp_bin_{i}' for i in range(10)]
plot_step1_multi_bar(df_stats, lbp_keys, "Micro-Texture (LBP Bins)")

In [ ]:
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def radar_plot(df_stats, feature_keys, category_name, max_classes=5):
    target_cols = [f"{feat}_mean" for feat in feature_keys]
    labels = [feat.replace('_', ' ').title() for feat in feature_keys]
    
    # --- STEP 1: TRANSFORMATION & SCALING ---
    pt = PowerTransformer(method='yeo-johnson')
    mms = MinMaxScaler(feature_range=(0.1, 0.9)) 
    
    data_transformed = pt.fit_transform(df_stats[target_cols])
    data_final = mms.fit_transform(data_transformed)
    df_norm = pd.DataFrame(data_final, index=df_stats.index, columns=target_cols)
    
    # --- STEP 2: COLOR MAPPING ---
    all_classes = df_norm.index.tolist()
    palette = sns.color_palette("husl", len(all_classes))
    color_map = dict(zip(all_classes, palette))
    
    class_groups = [all_classes[i:i + max_classes] for i in range(0, len(all_classes), max_classes)]
    
    num_vars = len(labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    for group_idx, selected_classes in enumerate(class_groups):
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
        
        # Plot data utama
        for cls in selected_classes:
            values = df_norm.loc[cls].values.tolist()
            values += values[:1]
            # Zorder 5 agar data berada di bawah angka
            ax.plot(angles, values, linewidth=2, label=cls, color=color_map[cls], 
                    marker='o', markersize=4, zorder=5)
            ax.fill(angles, values, color=color_map[cls], alpha=0.1, zorder=4)

        # Pengaturan Grid Dasar
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11, fontweight='bold')
        
        # --- PENEMPATAN ANGKA DI SETIAP SUMBU (DIPERTEGAS) ---
        grid_values = [0.2, 0.4, 0.6, 0.8]
        for angle in angles[:-1]:
            for r in grid_values:
                # Menggunakan zorder=10 agar teks berada di paling depan
                ax.text(angle, r, f"{r}", color="black", size=8, 
                        ha='center', va='center', fontweight='bold',
                        zorder=10, 
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        
        ax.set_yticklabels([]) # Hapus label default
        ax.set_ylim(0, 1)
        ax.grid(True, color='#cccccc', linestyle='--', alpha=0.6)
        
        plt.title(f"Radar Signature: {category_name}\n(Group {group_idx + 1})", 
                  y=1.1, fontweight='bold', fontsize=15)
        plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), frameon=True, shadow=True)
        
        plt.tight_layout()
        plt.show()

In [ ]:
# --- DEFINISI KATEGORI SESUAI PILAR METODOLOGI ---



# 1. Non-Euclidean Complexity

cat_noneuc = ['fractal_dimension', 'lacunarity', 'entropy']



# 2. Spatial Texture (GLCM)

cat_glcm = ['glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']



# 3. Intensity & Histogram Stats

cat_intensity = ['mean_intensity', 'std_intensity', 'skewness', 'kurtosis', 'bimodal_coeff']



# 4. Morphology (Object Geometry)

cat_morph = ['solidity', 'aspect_ratio']



# 5. Frequency & Quality

cat_quality = ['spectral_entropy', 'blur_score', 'mean_gradient']



# 6. Micro-Texture (LBP Bins - Dibagi 2 agar tidak terlalu padat)

cat_lbp_a = [f'lbp_bin_{i}' for i in range(5)]

cat_lbp_b = [f'lbp_bin_{i}' for i in range(5, 10)]

In [ ]:
# Jalankan untuk kategori utama Non-Euclidean
radar_plot(df_stats, cat_noneuc, "Non-Euclidean Complexity")

# Jalankan untuk kategori GLCM
radar_plot(df_stats, cat_glcm, "Spatial Texture (GLCM)")
    
# Jalankan untuk LBP Bins
radar_plot(df_stats, cat_lbp_a, "Micro-Texture LBP (Part A)")

radar_plot(df_stats, cat_lbp_b, "Micro-Texture LBP (Part B)")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def plot_univariate_analysis(df, features, n_cols=3):
    """
    Plotting univariate untuk melihat distribusi, kerapatan, dan outlier.
    Mendukung pilar EDA Komprehensif.
    """
    n_rows = (len(features) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(features):
        # 1. Boxplot untuk melihat Outlier dan Kuartil
        sns.boxplot(data=df, x='class', y=col, ax=axes[i], hue='class', 
                    legend=False, palette='viridis', fliersize=4)
        
        # 2. Stripplot untuk melihat Sebaran/Distribusi data asli (titik)
        sns.stripplot(data=df, x='class', y=col, ax=axes[i], 
                      color='black', alpha=0.3, size=2)
        
        axes[i].set_title(f'Distribusi & Outlier: {col}', fontsize=12, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=45)

    # Hapus axes yang tidak terpakai
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

# Definisikan kelompok fitur termasuk LBP Bins
texture_features = ['fractal_dimension', 'lacunarity', 'glcm_contrast', 'glcm_homogeneity', 'entropy']
intensity_features = ['mean_intensity', 'std_intensity', 'skewness', 'kurtosis', 'cv_intensity', 'bimodal_coeff']
quality_morph_features = ['blur_score', 'spectral_entropy', 'mean_gradient', 'solidity', 'aspect_ratio']
lbp_features = [f'lbp_bin_{i}' for i in range(10)]

# Eksekusi Plotting
print("1. Analisis Univariate: Fitur Tekstur (Non-Euclidean & Global)")
plot_univariate_analysis(df_features, texture_features)

print("2. Analisis Univariate: Fitur Intensitas & Statistik Histogram")
plot_univariate_analysis(df_features, intensity_features)

print("3. Analisis Univariate: Fitur Kualitas, Morfologi, & Frekuensi")
plot_univariate_analysis(df_features, quality_morph_features)

print("4. Analisis Univariate: Fitur LBP Bins (Mikro-Tekstur)")
plot_univariate_analysis(df_features, lbp_features)

## Ringkasan outlier univariate pada semua fitur awal

Bagian ini menghitung flag outlier berbasis IQR untuk setiap fitur awal dan setiap kelas. Analisis ini dilakukan sebelum ANOVA, IG/MI, dan Hybrid PCA. Hasilnya bersifat deskriptif; sampel tidak dihapus otomatis.

In [ ]:

# ============================================================
# UNIVARIATE OUTLIER SUMMARY PADA SEMUA FITUR AWAL
# ============================================================
# Analisis ini dilakukan sebelum seleksi ANOVA + IG/MI dan sebelum Hybrid PCA.
# Tujuannya adalah membaca fitur awal mana yang memiliki nilai ekstrem per kelas.
# Hasilnya bersifat deskriptif dan tidak digunakan untuk menghapus sampel otomatis.

def compute_univariate_outlier_flags(df, features, group_col="class", iqr_multiplier=1.5):
    work = df.copy()
    summary_rows = []
    flag_cols = []

    for feat in features:
        if feat not in work.columns or not pd.api.types.is_numeric_dtype(work[feat]):
            continue

        flag_col = f"{feat}_iqr_outlier"
        flag_cols.append(flag_col)
        work[flag_col] = False

        for cls, g in work.groupby(group_col):
            series = g[feat].replace([np.inf, -np.inf], np.nan).dropna()
            if len(series) < 4:
                continue

            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - iqr_multiplier * iqr
            upper = q3 + iqr_multiplier * iqr

            idx = g[(g[feat] < lower) | (g[feat] > upper)].index
            work.loc[idx, flag_col] = True

            summary_rows.append({
                "feature": feat,
                "class": cls,
                "n_samples": int(len(g)),
                "q1": float(q1),
                "q3": float(q3),
                "iqr": float(iqr),
                "lower_bound": float(lower),
                "upper_bound": float(upper),
                "n_iqr_outliers": int(len(idx)),
                "outlier_ratio": float(len(idx) / max(len(g), 1)),
            })

    work["univariate_iqr_outlier_count_raw_features"] = work[flag_cols].sum(axis=1) if flag_cols else 0
    summary = pd.DataFrame(summary_rows).sort_values(
        ["outlier_ratio", "n_iqr_outliers", "feature"],
        ascending=[False, False, True]
    )
    return work, summary, flag_cols

raw_feature_cols_for_outlier = get_numeric_feature_columns(df_features)
df_features_univariate_outliers, univariate_outlier_summary, univariate_flag_cols = compute_univariate_outlier_flags(
    df_features,
    raw_feature_cols_for_outlier,
    group_col="class",
    iqr_multiplier=1.5,
)

print(f"Jumlah fitur awal yang dianalisis secara univariate: {len(raw_feature_cols_for_outlier)}")
print(f"Jumlah kolom flag outlier IQR yang dibuat: {len(univariate_flag_cols)}")
display(univariate_outlier_summary.head(30))

univariate_outlier_summary.to_csv("univariate_outlier_summary_all_raw_features.csv", index=False)
df_features_univariate_outliers[["original_path", "class", "univariate_iqr_outlier_count_raw_features"]].to_csv(
    "univariate_outlier_counts_per_image_raw_features.csv",
    index=False,
)

plt.figure(figsize=(12, 5))
sns.barplot(
    data=(
        df_features_univariate_outliers.groupby("class")["univariate_iqr_outlier_count_raw_features"]
        .mean()
        .reset_index()
        .sort_values("univariate_iqr_outlier_count_raw_features", ascending=False)
    ),
    x="class",
    y="univariate_iqr_outlier_count_raw_features",
)
plt.xticks(rotation=45, ha="right")
plt.title("Rata-rata Jumlah Flag Outlier Univariate per Kelas pada Fitur Awal")
plt.xlabel("Kelas")
plt.ylabel("Rata-rata flag outlier IQR")
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_pearson_heatmap(df, features_list, title="Correlation Heatmap"):
    """Heatmap Pearson dengan ukuran dinamis dan nama file aman."""
    n_features = len(features_list)
    side_length = max(8, n_features * 0.6)
    figsize = (side_length + 2, side_length)

    if n_features <= 10:
        font_size, annot_size = 10, 9
    elif n_features <= 20:
        font_size, annot_size = 8, 7
    else:
        font_size = min(7, 200 / n_features)
        annot_size = max(5, font_size - 1)

    corr_matrix = df[features_list].corr(method="pearson")
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    plt.figure(figsize=figsize)
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True if n_features <= 35 else False,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        linewidths=.5,
        cbar_kws={"shrink": .8},
        annot_kws={"size": annot_size}
    )

    plt.title(title, fontsize=14, fontweight="bold", pad=20)
    plt.xticks(rotation=45, ha="right", fontsize=font_size)
    plt.yticks(fontsize=font_size)
    plt.tight_layout()

    file_name = f"{sanitize_filename(title)}_{n_features}f.png"
    plt.savefig(file_name, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Plot ({n_features} fitur) disimpan sebagai: {file_name} dengan figsize {figsize}")


In [ ]:
target_features = get_numeric_feature_columns(df_features)
plot_pearson_heatmap(df_features, target_features, title="Tahap 1 - Initial Screening Heatmap")

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def diagnose_multicollinearity(df, features):
    X = df[features].replace([np.inf, -np.inf], np.nan).dropna(axis=0)
    X = X.loc[:, X.std(numeric_only=True) > 0]

    corr_matrix = X.corr()
    eigenvalues = np.linalg.eigvals(corr_matrix)
    eigenvalues = np.real(eigenvalues)
    min_ev = max(np.min(eigenvalues), 1e-12)
    cond_number = np.sqrt(max(eigenvalues) / min_ev)

    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

    print("--- DIAGNOSIS GLOBAL ---")
    print(f"Condition Number: {cond_number:.2f}")
    print(f"Eigenvalues Terendah: {np.min(eigenvalues):.6f}")
    print("\n--- DIAGNOSIS LOKAL (VIF) ---")
    return vif_data.sort_values(by="VIF", ascending=False)

target_features = get_numeric_feature_columns(df_features)
vif_summary = diagnose_multicollinearity(df_features, target_features)
display(vif_summary)


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

def iterative_vif_pruning(df, features, threshold=10.0, min_features=3):
    """Eliminasi fitur dengan VIF tinggi, dengan guard agar tidak membuang terlalu banyak fitur."""
    current_features = list(features)
    iteration = 1

    while len(current_features) > min_features:
        X = df[current_features].replace([np.inf, -np.inf], np.nan).dropna(axis=0)
        X = X.loc[:, X.std(numeric_only=True) > 0]
        current_features = [f for f in current_features if f in X.columns]

        if len(current_features) <= min_features:
            break

        vif_values = [variance_inflation_factor(X.values, i) for i in range(len(current_features))]
        vif_df = pd.DataFrame({"Feature": current_features, "VIF": vif_values})

        max_vif = vif_df["VIF"].max()
        if not np.isfinite(max_vif) or max_vif > threshold:
            excluded_feature = vif_df.sort_values("VIF", ascending=False).iloc[0]["Feature"]
            current_features.remove(excluded_feature)
            print(f"Iterasi {iteration}: Membuang '{excluded_feature}' (VIF: {max_vif:.2f})")
            iteration += 1
        else:
            break

    final_corr = df[current_features].corr()
    final_eigen = np.real(np.linalg.eigvals(final_corr))
    min_ev = max(np.min(final_eigen), 1e-12)
    final_cond = np.sqrt(max(final_eigen) / min_ev)

    print("\n--- HASIL AKHIR ---")
    print(f"Fitur Tersisa: {len(current_features)}")
    print(f"Condition Number Baru: {final_cond:.2f}")
    print(current_features)
    return current_features

features_to_prune = get_numeric_feature_columns(df_features)
final_feature_candidates = iterative_vif_pruning(df_features, features_to_prune)


In [ ]:
# 'final_feature_candidates' adalah hasil dari fungsi pruning Anda sebelumnya
plot_pearson_heatmap(df_features, final_feature_candidates, title="Tahap 4 - Post Pruning Validation Heatmap")

In [ ]:
from sklearn.feature_selection import f_classif
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.feature_selection import f_classif
import pandas as pd

def calculate_anova_metrics(X, y):
    """
    Modul Kalkulasi: Mengambil nilai signifikansi statistik fitur.
    X: DataFrame berisi fitur saja (tanpa kolom class).
    y: Series/Array berisi label kelas.
    """
    # Mengambil nama kolom dari X untuk index DataFrame hasil
    feature_names = X.columns.tolist()
    
    # Perhitungan ANOVA F-value
    f_scores, p_values = f_classif(X, y)
    
    # Pembuatan DataFrame hasil yang rapi
    anova_df = pd.DataFrame({
        'Feature': feature_names,
        'F_Score': f_scores,
        'P_Value': p_values
    }).sort_values(by='F_Score', ascending=False)
    
    return anova_df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def render_anova_plot(anova_df, title="ANOVA F-Score Ranking"):
    """
    Modul Visualisasi: Membangun representasi visual daya diskriminatif fitur.
    Pilar 4: Standardisasi XAI.
    """
    n_features = len(anova_df)
    plt.figure(figsize=(12, max(6, n_features * 0.4)))
    
    # Pemilihan palet warna sesuai standar publikasi IEEE
    colors = sns.color_palette('viridis', n_colors=n_features)
    
    ax = sns.barplot(
        data=anova_df, 
        x='F_Score', 
        y='Feature', 
        hue='F_Score', 
        palette=colors, 
        legend=False, 
        edgecolor='black', 
        alpha=0.8
    )
    
    # Anotasi nilai F-Score di ujung bar
    max_score = anova_df['F_Score'].max()
    for i, score in enumerate(anova_df['F_Score']):
        plt.text(
            score + (max_score * 0.01), 
            i, 
            f'{score:.2f}', 
            va='center', 
            fontsize=9, 
            fontweight='bold'
        )

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("F-Score (Daya Diskriminatif Linear)")
    plt.ylabel("Daftar Fitur")
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Siapkan X (Fitur terpilih) dan y (Target)
X_vif = df_features[final_feature_candidates]
y_labels = df_features['class']

# 2. Hitung
anova_vif = calculate_anova_metrics(X_vif, y_labels)

# 3. Visualisasi
render_anova_plot(anova_vif, title="ANOVA Ranking: VIF Pruned Features")

pd.DataFrame(anova_vif).to_csv("anova_VIF Pruning_results.csv", index=False)


## Seleksi fitur konsensus: Top ANOVA ∩ Top Information Gain/Mutual Information

Tahap ini menggunakan dua metode univariate yang saling melengkapi. **ANOVA F-score** dipakai sebagai baseline statistik untuk membaca separabilitas fitur berdasarkan perbedaan rerata antarkelas. **Information Gain** pada fitur kontinu diestimasi menggunakan `mutual_info_classif` untuk membaca ketergantungan informasi antara fitur dan label, termasuk pola non-linear yang tidak selalu tertangkap oleh ANOVA.

Karena distribusi kelas final setelah deduplikasi dapat tidak sepenuhnya seimbang, ranking ANOVA dan Information Gain dihitung menggunakan **balanced bootstrap per kelas**. Pada setiap iterasi, setiap kelas diambil dengan jumlah sampel yang sama, sehingga ranking fitur tidak terlalu didominasi oleh kelas yang memiliki jumlah sampel lebih besar.

Aturan seleksi yang digunakan:

1. Ambil **Top-k ANOVA** dan **Top-k Information Gain/Mutual Information**.
2. Fitur yang muncul di kedua daftar ditetapkan sebagai **Core Stable Features**.
3. Fitur lain yang tidak masuk irisan tidak langsung dibuang, tetapi diperlakukan sebagai **Residual Features**.
4. Residual features diringkas menggunakan **Hybrid PCA** agar informasi yang masih berguna tetap dipertahankan tanpa membawa redundansi fitur yang berlebihan.

Dengan demikian, feature set akhir dibangun sebagai:

```text
Final EDA Feature Set = Core Stable Features + PCA(Residual Features)
```

Jika terdapat 5 fitur yang muncul sekaligus pada Top ANOVA dan Top IG/MI, maka kelima fitur tersebut menjadi fitur inti utama. Jika irisan kurang dari 5, jumlah fitur inti tidak dipaksakan; hasil tersebut dilaporkan apa adanya agar interpretasi tetap defensible.


In [ ]:
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler


def _prepare_feature_matrix(df, features, target_col="class"):
    valid_features = [f for f in features if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]
    X = df[valid_features].replace([np.inf, -np.inf], np.nan).copy()
    X = X.apply(lambda s: s.fillna(s.median()), axis=0)
    # Buang fitur konstan karena tidak informatif untuk ANOVA maupun MI.
    non_constant = X.columns[X.std(axis=0) > 0].tolist()
    X = X[non_constant]
    y = df[target_col].astype(str)
    return X, y


def balanced_anova_mi_ranking(
    df,
    features,
    target_col="class",
    n_bootstrap=30,
    max_per_class=None,
    random_state=42,
    mi_neighbors=3,
):
    """Membandingkan ANOVA dan Information Gain/Mutual Information secara class-balanced.

    Setiap bootstrap mengambil jumlah sampel yang sama dari setiap kelas. Ini mengurangi bias
    ranking fitur ketika distribusi kelas final tidak seimbang.
    """
    rng = np.random.default_rng(random_state)
    X_all, y_all = _prepare_feature_matrix(df, features, target_col=target_col)
    work_df = X_all.copy()
    work_df[target_col] = y_all.values

    class_counts = work_df[target_col].value_counts()
    min_count = int(class_counts.min())
    n_per_class = min_count if max_per_class is None else min(min_count, int(max_per_class))

    if n_per_class < 3:
        raise ValueError("Setiap kelas membutuhkan minimal 3 sampel untuk balanced bootstrap.")

    rows = []
    for b in range(n_bootstrap):
        sampled_parts = []
        for cls, group in work_df.groupby(target_col):
            sampled_parts.append(
                group.sample(
                    n=n_per_class,
                    replace=False,
                    random_state=int(rng.integers(0, 1_000_000)),
                )
            )
        boot_df = pd.concat(sampled_parts, ignore_index=True)

        X_boot = boot_df[X_all.columns]
        y_boot = boot_df[target_col]

        f_scores, p_values = f_classif(X_boot, y_boot)
        f_scores = np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0)

        X_scaled = StandardScaler().fit_transform(X_boot)
        mi_scores = mutual_info_classif(
            X_scaled,
            y_boot,
            discrete_features=False,
            n_neighbors=mi_neighbors,
            random_state=random_state + b,
        )
        mi_scores = np.nan_to_num(mi_scores, nan=0.0, posinf=0.0, neginf=0.0)

        f_rank = pd.Series(f_scores, index=X_all.columns).rank(ascending=False, method="average")
        mi_rank = pd.Series(mi_scores, index=X_all.columns).rank(ascending=False, method="average")

        # Normalisasi 0-1 per bootstrap agar skala ANOVA dan MI dapat disejajarkan untuk skor konsensus.
        f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-12)
        mi_norm = (mi_scores - mi_scores.min()) / (mi_scores.max() - mi_scores.min() + 1e-12)

        for i, feat in enumerate(X_all.columns):
            rows.append({
                "bootstrap": b,
                "Feature": feat,
                "ANOVA_F": float(f_scores[i]),
                "ANOVA_P": float(p_values[i]) if np.isfinite(p_values[i]) else np.nan,
                "Information_Gain_MI": float(mi_scores[i]),
                "ANOVA_Rank": float(f_rank.loc[feat]),
                "IG_Rank": float(mi_rank.loc[feat]),
                "ANOVA_Norm": float(f_norm[i]),
                "IG_Norm": float(mi_norm[i]),
            })

    raw = pd.DataFrame(rows)
    summary = raw.groupby("Feature").agg(
        ANOVA_F_Mean=("ANOVA_F", "mean"),
        ANOVA_F_Std=("ANOVA_F", "std"),
        Information_Gain_Mean=("Information_Gain_MI", "mean"),
        Information_Gain_Std=("Information_Gain_MI", "std"),
        ANOVA_Rank_Mean=("ANOVA_Rank", "mean"),
        IG_Rank_Mean=("IG_Rank", "mean"),
        ANOVA_Norm_Mean=("ANOVA_Norm", "mean"),
        IG_Norm_Mean=("IG_Norm", "mean"),
    ).reset_index()

    summary["Consensus_Rank"] = (summary["ANOVA_Rank_Mean"] + summary["IG_Rank_Mean"]) / 2
    summary["Consensus_Score"] = (summary["ANOVA_Norm_Mean"] + summary["IG_Norm_Mean"]) / 2

    def interpret(row):
        a = row["ANOVA_Rank_Mean"]
        m = row["IG_Rank_Mean"]
        if a <= 5 and m <= 5:
            return "robust_linear_and_nonlinear"
        if a <= 5 and m > 5:
            return "mainly_mean_difference"
        if a > 5 and m <= 5:
            return "possible_nonlinear_dependency"
        return "secondary_feature"

    summary["Interpretation"] = summary.apply(interpret, axis=1)
    summary = summary.sort_values(["Consensus_Rank", "Feature"]).reset_index(drop=True)

    meta = {
        "n_bootstrap": n_bootstrap,
        "n_per_class": n_per_class,
        "class_counts": class_counts.to_dict(),
        "features_used": list(X_all.columns),
    }
    return summary, raw, meta


def plot_anova_ig_comparison(summary_df, top_n=20, title="Balanced ANOVA vs Information Gain Ranking"):
    plot_df = summary_df.head(top_n).copy()
    long_df = plot_df.melt(
        id_vars="Feature",
        value_vars=["ANOVA_Norm_Mean", "IG_Norm_Mean"],
        var_name="Metric",
        value_name="Normalized Score",
    )
    metric_labels = {
        "ANOVA_Norm_Mean": "ANOVA F-score",
        "IG_Norm_Mean": "Information Gain / MI",
    }
    long_df["Metric"] = long_df["Metric"].map(metric_labels)

    plt.figure(figsize=(12, max(6, top_n * 0.35)))
    sns.barplot(data=long_df, x="Normalized Score", y="Feature", hue="Metric")
    plt.title(title)
    plt.xlabel("Rerata skor ternormalisasi dari balanced bootstrap")
    plt.ylabel("Fitur")
    plt.grid(axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


def select_core_residual_features(summary_df, all_features, top_k=10, fallback_min_core=1):
    """Menentukan core features dari irisan Top-k ANOVA dan Top-k IG/MI.

    - Core Stable Features = Top-k ANOVA ∩ Top-k IG/MI.
    - Residual Features = semua fitur numerik lain yang tidak masuk core.
    - Jika irisan kosong, fallback teknis memakai fitur consensus tertinggi agar pipeline PCA tetap berjalan.
      Fallback ini harus dilaporkan sebagai kondisi teknis, bukan klaim bahwa fitur tersebut stabil pada dua metode.
    """
    ranked_anova = summary_df.sort_values(["ANOVA_Rank_Mean", "Feature"])
    ranked_ig = summary_df.sort_values(["IG_Rank_Mean", "Feature"])
    ranked_consensus = summary_df.sort_values(["Consensus_Rank", "Feature"])

    top_anova = ranked_anova.head(top_k)["Feature"].tolist()
    top_ig = ranked_ig.head(top_k)["Feature"].tolist()
    overlap_set = set(top_anova).intersection(top_ig)

    core_features = [f for f in ranked_consensus["Feature"].tolist() if f in overlap_set]
    fallback_used = False
    if len(core_features) == 0 and fallback_min_core > 0:
        fallback_used = True
        core_features = ranked_consensus.head(min(fallback_min_core, len(ranked_consensus)))["Feature"].tolist()

    all_features_ordered = [f for f in all_features if f in summary_df["Feature"].values]
    residual_features = [f for f in all_features_ordered if f not in core_features]

    selection_table = summary_df.copy()
    selection_table["Selection_Group"] = "Residual_Hybrid_PCA"
    selection_table.loc[selection_table["Feature"].isin(core_features), "Selection_Group"] = "Core_Stable_ANOVA_and_IG"
    selection_table.loc[selection_table["Feature"].isin(set(top_anova) - overlap_set), "Selection_Group"] = "Top_ANOVA_Only_Residual"
    selection_table.loc[selection_table["Feature"].isin(set(top_ig) - overlap_set), "Selection_Group"] = "Top_IG_Only_Residual"
    if fallback_used:
        selection_table.loc[selection_table["Feature"].isin(core_features), "Selection_Group"] = "Fallback_Consensus_Core"

    meta = {
        "top_k": top_k,
        "top_anova": top_anova,
        "top_ig": top_ig,
        "core_features": core_features,
        "residual_features": residual_features,
        "n_core_features": len(core_features),
        "n_residual_features": len(residual_features),
        "fallback_used": fallback_used,
    }
    return core_features, residual_features, selection_table, meta


# Perbandingan ANOVA vs IG/MI pada semua fitur numerik hasil ekstraksi.
target_features = get_numeric_feature_columns(df_features)
balanced_rank_all, balanced_rank_all_raw, balanced_rank_meta = balanced_anova_mi_ranking(
    df_features,
    target_features,
    n_bootstrap=30,
    random_state=42,
    mi_neighbors=3,
)

print("Metadata balanced ranking seluruh fitur numerik:")
print(balanced_rank_meta)
display(balanced_rank_all)
plot_anova_ig_comparison(
    balanced_rank_all,
    top_n=min(20, len(balanced_rank_all)),
    title="Balanced ANOVA vs IG/MI: All Numeric EDA Features",
)

TOP_K_UNIVARIATE = 10
core_stable_features, residual_pca_features, feature_selection_table, feature_selection_meta = select_core_residual_features(
    balanced_rank_all,
    target_features,
    top_k=TOP_K_UNIVARIATE,
    fallback_min_core=1,
)

print("Top-k ANOVA:", feature_selection_meta["top_anova"])
print("Top-k IG/MI:", feature_selection_meta["top_ig"])
print("Core Stable Features (Top ANOVA ∩ Top IG/MI):", core_stable_features)
print("Jumlah core features:", feature_selection_meta["n_core_features"])
print("Jumlah residual features untuk Hybrid PCA:", feature_selection_meta["n_residual_features"])
if feature_selection_meta["fallback_used"]:
    print("PERINGATAN: Irisan Top ANOVA dan Top IG/MI kosong. Fallback consensus digunakan agar pipeline tetap berjalan.")

display(feature_selection_table.sort_values(["Selection_Group", "Consensus_Rank", "Feature"]))

balanced_rank_all.to_csv("balanced_anova_information_gain_all_features.csv", index=False)
balanced_rank_all_raw.to_csv("balanced_anova_information_gain_all_features_raw_bootstrap.csv", index=False)
feature_selection_table.to_csv("core_stable_features_anova_ig_selection_table.csv", index=False)
pd.DataFrame({"Core_Stable_Feature": core_stable_features}).to_csv("core_stable_features_anova_ig.csv", index=False)
pd.DataFrame({"Residual_Hybrid_PCA_Feature": residual_pca_features}).to_csv("residual_features_for_hybrid_pca.csv", index=False)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


def apply_hybrid_pca(df, clean_features, residual_features, n_components=3):
    """Membangun final EDA feature set.

    clean_features adalah Core Stable Features dari irisan Top ANOVA dan Top IG/MI.
    residual_features adalah fitur lain yang tidak masuk core dan diringkas dengan PCA.
    """
    clean_features = [f for f in clean_features if f in df.columns]
    residual_features = [f for f in residual_features if f in df.columns and f not in clean_features]

    if len(clean_features) == 0:
        raise ValueError("Core stable features kosong. Jalankan seleksi ANOVA ∩ IG/MI terlebih dahulu atau aktifkan fallback.")

    X_clean = df[clean_features].replace([np.inf, -np.inf], np.nan).copy()
    X_clean = X_clean.apply(lambda s: s.fillna(s.median()), axis=0)

    if len(residual_features) == 0:
        print("Tidak ada residual feature untuk PCA. Mengembalikan core stable features saja.")
        return X_clean, None, None, pd.DataFrame()

    n_components = min(n_components, len(residual_features))
    X_residual = df[residual_features].replace([np.inf, -np.inf], np.nan)
    X_residual = X_residual.apply(lambda s: s.fillna(s.median()), axis=0)

    scaler = StandardScaler()
    X_residual_scaled = scaler.fit_transform(X_residual)

    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_residual_scaled)

    pca_cols = [f"PC_Residual_{i+1}" for i in range(n_components)]
    df_pca = pd.DataFrame(X_pca, columns=pca_cols, index=df.index)

    df_hybrid = pd.concat([X_clean, df_pca], axis=1)

    loadings = pd.DataFrame(
        pca.components_.T,
        index=residual_features,
        columns=pca_cols,
    )

    print("Core Stable Features:", clean_features)
    print("Jumlah residual features yang diringkas PCA:", len(residual_features))
    print(f"Informasi residual yang terjaga di PCA: {pca.explained_variance_ratio_.sum()*100:.2f}%")

    return df_hybrid, pca, scaler, loadings


PCA_N_COMPONENTS = 3
# Final EDA feature set = Core Stable Features + PCA(Residual Features)
df_hybrid_final, pca_model, pca_scaler, pca_residual_loadings = apply_hybrid_pca(
    df_features,
    core_stable_features,
    residual_pca_features,
    n_components=PCA_N_COMPONENTS,
)

# Simpan rancangan fitur akhir agar dapat dilaporkan di Bab IV.
hybrid_feature_design = pd.DataFrame({
    "Feature": list(df_hybrid_final.columns),
    "Role": ["Core_Stable_Feature" if f in core_stable_features else "PCA_Residual_Component" for f in df_hybrid_final.columns],
})

display(hybrid_feature_design)
display(pca_residual_loadings if pca_model is not None else pd.DataFrame())

hybrid_feature_design.to_csv("hybrid_pca_final_feature_design.csv", index=False)
if pca_model is not None:
    pca_residual_loadings.to_csv("hybrid_pca_residual_loadings.csv")


## Output final feature set untuk notebook 3

Blok ini wajib dijalankan setelah `df_hybrid_final` terbentuk. Tujuannya adalah membuat file **sample-level** yang akan dibaca oleh notebook ketiga untuk analisis outlier multivariat.

File yang dihasilkan:

```text
final_feature_set_for_multivariate_outlier.csv
hybrid_pca_final_feature_matrix.csv
final_feature_set_metadata.csv
```

Isi file utama mencakup metadata sampel dari `dataset_final_manifest.csv` dan fitur akhir dari notebook kedua:

```text
path, filename, class, original_path, Core Stable Features, PC_Residual_*
```

Dengan demikian, notebook ketiga tidak perlu membaca ulang semua fitur awal dan tidak perlu menjalankan ANOVA/IG lagi. Notebook ketiga cukup memakai final feature set ini untuk menghitung outlier multivariat secara deskriptif.


In [ ]:
# ============================================================
# EXPORT FINAL FEATURE SET UNTUK NOTEBOOK 3
# ============================================================
# Notebook 3 akan membaca file ini untuk multivariate descriptive outlier detection.
# Fitur yang dibawa hanya final feature set:
# Core Stable Features + PC_Residual dari Hybrid PCA.

FINAL_FEATURE_SET_CSV = Path("final_feature_set_for_multivariate_outlier.csv")
HYBRID_FEATURE_MATRIX_CSV = Path("hybrid_pca_final_feature_matrix.csv")
FINAL_FEATURE_METADATA_CSV = Path("final_feature_set_metadata.csv")


def export_final_feature_set_for_outlier(
    df_manifest,
    df_features,
    df_hybrid_final,
    core_features,
    residual_features,
    pca_model=None,
    output_csv=FINAL_FEATURE_SET_CSV,
    matrix_csv=HYBRID_FEATURE_MATRIX_CSV,
    metadata_csv=FINAL_FEATURE_METADATA_CSV,
):
    """Menyimpan final feature set sample-level untuk notebook post-processing/outlier.

    Parameter
    ---------
    df_manifest : DataFrame
        Manifest final dari notebook 1. Wajib memiliki kolom path, class, filename.
    df_features : DataFrame
        Hasil ekstraksi fitur pada notebook 2. Wajib memiliki kolom original_path dan class.
    df_hybrid_final : DataFrame
        Final feature set hasil Core Stable Features + Hybrid PCA Residual.
    core_features : list[str]
        Fitur inti hasil irisan Top ANOVA dan Top IG/MI.
    residual_features : list[str]
        Fitur residual sebelum diringkas dengan PCA.
    pca_model : sklearn.decomposition.PCA | None
        Model PCA residual. Dipakai untuk metadata explained variance.

    Output
    ------
    final_feature_set_for_multivariate_outlier.csv
        File utama untuk notebook 3. Berisi metadata sampel + fitur final.
    hybrid_pca_final_feature_matrix.csv
        Alias/fallback dengan isi sama, agar kompatibel dengan notebook 3.
    final_feature_set_metadata.csv
        Ringkasan struktur fitur final dan metadata PCA.
    """
    required_manifest_cols = {"path", "class", "filename"}
    missing_manifest = required_manifest_cols - set(df_manifest.columns)
    if missing_manifest:
        raise ValueError(f"df_manifest tidak memiliki kolom wajib: {missing_manifest}")

    if "original_path" not in df_features.columns:
        raise ValueError("df_features wajib memiliki kolom original_path dari proses ekstraksi fitur.")

    if len(df_features) != len(df_hybrid_final):
        raise ValueError(
            f"Jumlah baris df_features ({len(df_features)}) berbeda dari df_hybrid_final ({len(df_hybrid_final)}). "
            "Pastikan df_hybrid_final dibuat dari df_features yang sama."
        )

    # Metadata dari manifest final notebook 1.
    meta_manifest = df_manifest[["path", "filename", "class"]].copy()
    meta_manifest["path"] = meta_manifest["path"].astype(str)
    meta_manifest["_path_key"] = meta_manifest["path"].astype(str)

    # Metadata dari hasil feature extraction notebook 2.
    meta_features = df_features[["original_path", "class"]].copy()
    meta_features["original_path"] = meta_features["original_path"].astype(str)
    meta_features["_path_key"] = meta_features["original_path"].astype(str)

    # Validasi bahwa path feature berasal dari manifest final.
    manifest_paths = set(meta_manifest["_path_key"])
    feature_paths = set(meta_features["_path_key"])
    missing_in_manifest = sorted(feature_paths - manifest_paths)
    missing_in_features = sorted(manifest_paths - feature_paths)
    if missing_in_manifest or missing_in_features:
        raise ValueError(
            "Path pada df_features dan dataset_final_manifest.csv tidak konsisten. "
            f"Contoh path fitur tidak ada di manifest: {missing_in_manifest[:5]}; "
            f"contoh path manifest belum punya fitur: {missing_in_features[:5]}"
        )

    # Ikuti urutan df_features/df_hybrid_final agar alignment indeks tetap benar.
    meta_ordered = meta_features.merge(
        meta_manifest[["_path_key", "path", "filename"]],
        on="_path_key",
        how="left",
        validate="one_to_one",
    )

    # Gunakan class dari df_features sebagai class operasional, tetapi validasi dengan manifest.
    class_check = meta_ordered.merge(
        meta_manifest[["_path_key", "class"]].rename(columns={"class": "class_manifest"}),
        on="_path_key",
        how="left",
        validate="one_to_one",
    )
    mismatch = class_check[class_check["class"] != class_check["class_manifest"]]
    if not mismatch.empty:
        raise ValueError(
            "Ada mismatch label class antara df_features dan manifest final. "
            f"Contoh mismatch: {mismatch[['path', 'class', 'class_manifest']].head().to_dict('records')}"
        )

    final_features = df_hybrid_final.copy().reset_index(drop=True)
    final_feature_cols = final_features.columns.tolist()

    if not final_feature_cols:
        raise ValueError("df_hybrid_final tidak memiliki fitur final.")

    # Pastikan tidak ada NaN/inf pada fitur final.
    final_features = final_features.replace([np.inf, -np.inf], np.nan)
    final_features = final_features.apply(lambda s: s.fillna(s.median()), axis=0)

    df_out = pd.concat(
        [
            meta_ordered[["path", "filename", "class", "original_path"]].reset_index(drop=True),
            final_features.reset_index(drop=True),
        ],
        axis=1,
    )

    if len(df_out) != len(df_manifest):
        raise ValueError(
            f"Jumlah final feature set ({len(df_out)}) tidak sama dengan manifest final ({len(df_manifest)})."
        )

    # Simpan file utama dan fallback dengan isi sama.
    df_out.to_csv(output_csv, index=False)
    df_out.to_csv(matrix_csv, index=False)

    # Metadata fitur final untuk dokumentasi Bab IV.
    feature_roles = []
    for f in final_feature_cols:
        if f in core_features:
            role = "Core_Stable_Feature_ANOVA_and_IG"
        elif f.startswith("PC_Residual_"):
            role = "Hybrid_PCA_Residual_Component"
        else:
            role = "Final_Feature"
        feature_roles.append({"Feature": f, "Role": role})

    metadata_rows = [
        {"Key": "n_samples", "Value": len(df_out)},
        {"Key": "n_final_features", "Value": len(final_feature_cols)},
        {"Key": "n_core_stable_features", "Value": len(core_features)},
        {"Key": "n_residual_features_before_pca", "Value": len(residual_features)},
        {"Key": "final_feature_columns", "Value": ", ".join(final_feature_cols)},
        {"Key": "core_stable_features", "Value": ", ".join(core_features)},
        {"Key": "residual_features_before_pca", "Value": ", ".join(residual_features)},
    ]

    if pca_model is not None:
        metadata_rows.append({
            "Key": "pca_residual_explained_variance_ratio_sum",
            "Value": float(np.sum(pca_model.explained_variance_ratio_)),
        })
        metadata_rows.append({
            "Key": "pca_residual_explained_variance_ratio_each",
            "Value": ", ".join([f"{v:.6f}" for v in pca_model.explained_variance_ratio_]),
        })
    else:
        metadata_rows.append({"Key": "pca_residual_explained_variance_ratio_sum", "Value": "PCA not used"})

    df_metadata = pd.DataFrame(metadata_rows)
    df_roles = pd.DataFrame(feature_roles)
    df_metadata.to_csv(metadata_csv, index=False)
    df_roles.to_csv("final_feature_set_roles.csv", index=False)

    print("✅ Final feature set untuk notebook 3 berhasil dibuat.")
    print(f"File utama  : {output_csv} | {len(df_out)} sampel × {len(final_feature_cols)} fitur final")
    print(f"File fallback: {matrix_csv}")
    print(f"Metadata    : {metadata_csv}")
    print(f"Roles       : final_feature_set_roles.csv")

    display(df_out.head())
    display(df_roles)
    display(df_metadata)

    return df_out, df_roles, df_metadata


final_feature_set_for_outlier, final_feature_roles, final_feature_metadata = export_final_feature_set_for_outlier(
    df_manifest=df_manifest,
    df_features=df_features,
    df_hybrid_final=df_hybrid_final,
    core_features=core_stable_features,
    residual_features=residual_pca_features,
    pca_model=pca_model,
)


In [ ]:
plot_pearson_heatmap(df_hybrid_final, df_hybrid_final.columns.tolist(), title="Tahap 5 - Hybrid PCA Correlation Heatmap")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

def plot_pca_significance(pca):
    if pca is None:
        print("PCA tidak dijalankan karena tidak ada fitur redundan.")
        return

    exp_var = pca.explained_variance_ratio_
    cum_var = np.cumsum(exp_var)

    plt.figure(figsize=(10, 5))
    plt.bar(range(1, len(exp_var) + 1), exp_var, alpha=0.6, align="center", label="Individual Explained Variance")
    plt.step(range(1, len(cum_var) + 1), cum_var, where="mid", label="Cumulative Explained Variance", lw=2)
    plt.axhline(y=0.80, linestyle="--", label="80% Information Retained")
    plt.ylabel("Explained Variance Ratio")
    plt.xlabel("Principal Component Index")
    plt.title("Scree Plot: PCA Information Retention Analysis", fontsize=14, fontweight="bold")
    plt.legend(loc="best")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

plot_pca_significance(pca_model)


In [ ]:
def plot_pca_loadings(pca, residual_features):
    if pca is None:
        print("PCA tidak dijalankan, loading heatmap dilewati.")
        return

    loadings = pd.DataFrame(
        pca.components_.T,
        index=residual_features,
        columns=[f"PC_Residual_{i+1}" for i in range(pca.n_components_)],
    )

    plt.figure(figsize=(10, max(6, len(residual_features) * 0.28)))
    sns.heatmap(loadings, annot=False, cmap="coolwarm", center=0)
    plt.title("PCA Loadings pada Residual Features")
    plt.xlabel("Komponen PCA")
    plt.ylabel("Residual Feature")
    plt.tight_layout()
    plt.show()


plot_pca_loadings(pca_model, residual_pca_features)


In [ ]:
df_hybrid_final.columns.tolist()

In [ ]:
# df_hybrid_final berisi Core Stable Features + PC_Residual dari Hybrid PCA.
# Kita gunakan y_labels yang sama dari df_features asli.
X_hybrid = df_hybrid_final

# Hitung ANOVA pada feature set akhir sebagai sanity check statistik.
anova_hybrid = calculate_anova_metrics(X_hybrid, y_labels)

render_anova_plot(anova_hybrid, title="ANOVA Ranking: Core Stable + Hybrid PCA Residual Features")

pd.DataFrame(anova_hybrid).to_csv("anova_core_stable_plus_hybrid_pca_results.csv", index=False)


In [ ]:
# Perbandingan ANOVA vs IG/MI pada feature set akhir: Core Stable + PCA(Residual).
balanced_rank_hybrid, balanced_rank_hybrid_raw, balanced_rank_hybrid_meta = balanced_anova_mi_ranking(
    pd.concat([df_hybrid_final.reset_index(drop=True), df_features[["class"]].reset_index(drop=True)], axis=1),
    list(df_hybrid_final.columns),
    target_col="class",
    n_bootstrap=30,
    random_state=42,
    mi_neighbors=3,
)

print("Metadata balanced ranking final feature set:")
print(balanced_rank_hybrid_meta)
display(balanced_rank_hybrid)
plot_anova_ig_comparison(
    balanced_rank_hybrid,
    top_n=min(20, len(balanced_rank_hybrid)),
    title="Balanced ANOVA vs IG/MI: Core Stable + Hybrid PCA Residual Features",
)

balanced_rank_hybrid.to_csv("balanced_anova_information_gain_core_stable_hybrid_pca_features.csv", index=False)
balanced_rank_hybrid_raw.to_csv("balanced_anova_information_gain_core_stable_hybrid_pca_features_raw_bootstrap.csv", index=False)


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import Isomap
from sklearn.preprocessing import StandardScaler

def run_lda_projection(X, y, n_components=2):
    """Modul LDA: Supervised Dimensionality Reduction."""
    # LDA sangat sensitif terhadap skala, wajib standardisasi
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    lda = LDA(n_components=n_components)
    X_lda = lda.fit_transform(X_scaled, y)
    return X_lda, lda

def run_isomap_projection(X, n_components=2, n_neighbors=15):
    """Modul Isomap: Non-Linear Manifold Learning."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    iso = Isomap(n_components=n_components, n_neighbors=n_neighbors)
    X_iso = iso.fit_transform(X_scaled)
    return X_iso, iso

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def render_projection_plot(X_projected, y, title="Projection Plot", cmap='viridis'):
    """Modul Visualisasi: Merender hasil proyeksi ke scatter plot 2D."""
    plt.figure(figsize=(10, 7))
    
    # Gunakan scatterplot dengan gaya (style) berbeda per kelas agar ramah cetak hitam-putih
    sns.scatterplot(
        x=X_projected[:, 0], 
        y=X_projected[:, 1], 
        hue=y, 
        style=y,
        palette=cmap, 
        s=80, 
        alpha=0.7, 
        edgecolor='k'
    )
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Kelas Rempah')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- STEP 1: LDA (Supervised) ---
# Membuktikan separabilitas kelas secara linear maksimal
X_lda, lda_model = run_lda_projection(df_hybrid_final, y_labels)
render_projection_plot(X_lda, y_labels, title="LDA Projection: Class Separability (Hybrid PCA)")

# --- STEP 2: Isomap (Unsupervised Manifold) ---
# Membuktikan adanya struktur non-linear pada fitur tekstur
X_iso, iso_model = run_isomap_projection(df_hybrid_final)
render_projection_plot(X_iso, y_labels, title="Isomap Projection: Geodesic Manifold Structure", cmap='magma')

In [ ]:
from sklearn.manifold import TSNE
from sklearn.manifold import MDS

try:
    import trimap
    HAS_TRIMAP = True
except ImportError:
    HAS_TRIMAP = False
    print("⚠️ Paket 'trimap' tidak ditemukan. Benchmark TriMap akan dilewati.")


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import f_classif
import seaborn as sns
import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 150


In [ ]:
import time
import numpy as np
from joblib import Parallel, delayed
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.manifold import TSNE, MDS
from sklearn.preprocessing import StandardScaler

def execute_single_model(name, model, X_scaled, y):
    print(f"Starting execution: {name}...")
    start_time = time.time()
    try:
        X_2d = model.fit_transform(X_scaled)
        duration = time.time() - start_time

        s_score = silhouette_score(X_2d, y)
        ch_score = calinski_harabasz_score(X_2d, y)

        print(f"Finished {name} in {duration:.2f}s")
        return name, {
            "embedding": X_2d,
            "metrics": {
                "silhouette": s_score,
                "calinski_harabasz": ch_score,
                "time_seconds": duration
            }
        }
    except Exception as e:
        print(f"Error pada {name}: {str(e)}")
        return name, None

def run_manifold_benchmarks_parallel(X, y, random_state=42):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    models_dict = {
        "t-SNE": TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=random_state),
        "MDS": MDS(n_components=2, n_init=4, random_state=random_state),
    }

    if HAS_TRIMAP:
        models_dict["TriMap"] = trimap.TRIMAP(n_inliers=12, n_outliers=7, n_random=5)

    n_jobs = max(1, min(len(models_dict), (os.cpu_count() or 2) - 1))
    print(f"Launching parallel task execution with n_jobs={n_jobs}...")

    results_list = Parallel(n_jobs=n_jobs)(
        delayed(execute_single_model)(name, model, X_scaled, y)
        for name, model in models_dict.items()
    )

    return {name: data for name, data in results_list if data is not None}


In [ ]:
def generate_statistical_report(results):
    """
    Menghasilkan tabel ringkasan metrik untuk Bab 4.
    Fokus pada efisiensi (Time) dan kualitas pemisahan (Silhouette & CH).
    """
    summary_list = []
    for name, data in results.items():
        res = {
            "Algorithm": name,
            "Silhouette Score (↑)": data["metrics"]["silhouette"],
            "Calinski-Harabasz (↑)": data["metrics"]["calinski_harabasz"],
            "Exec Time (s) (↓)": data["metrics"]["time_seconds"]
        }
        summary_list.append(res)
    
    df_report = pd.DataFrame(summary_list).sort_values(by="Silhouette Score (↑)", ascending=False)
    return df_report




def render_manifold_gallery_optimized(results, y_labels):
    """
    Visualisasi grid manifold untuk naskah ilmiah (High DPI).
    Menampilkan metrik performa di setiap judul plot.
    """
    n_models = len(results)
    cols = 3
    rows = (n_models + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(21, 6 * rows), dpi=300)
    axes = axes.flatten() if n_models > 1 else [axes]

    for i, (name, data) in enumerate(results.items()):
        X_2d = data["embedding"]
        m = data["metrics"]
        
        # Scatter plot dengan palet warna Viridis (Standar IEEE)
        sns.scatterplot(
            x=X_2d[:, 0], y=X_2d[:, 1], 
            hue=y_labels, palette='viridis', 
            s=50, alpha=0.7, ax=axes[i], 
            edgecolor='w', linewidth=0.5,
            legend=(i == 0)
        )
        
        # Judul plot dengan rincian metrik
        axes[i].set_title(
            f"{name}\nSil: {m['silhouette']:.4f} | CH: {m['calinski_harabasz']:.1f}\nTime: {m['time_seconds']:.2f}s",
            fontweight='bold', fontsize=12
        )
        axes[i].axis('off')
        
        if i == 0:
            axes[i].legend(title="Kelas Rempah", bbox_to_anchor=(1.05, 1), loc='upper left')

    # Bersihkan axis kosong
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# 1. Jalankan Benchmark Paralel
# n_jobs=-1 akan memaksimalkan penggunaan OCPU di instance Oracle Anda
results = run_manifold_benchmarks_parallel(df_hybrid_final, y_labels)

# 2. Simpan hasil ke disk (Checkpointing - MLOps)
# Penting untuk persistensi data jika kernel atau koneksi terputus
import pickle
with open('manifold_results_checkpoint.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:
generate_statistical_report(results)


In [ ]:
render_manifold_gallery_optimized(results, y_labels)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

def calculate_class_feature_matrix(df, features, target_col='class'):
    """
    Modul Kalkulasi: Menghitung rata-rata fitur per kelas dan menormalisasinya.
    Pilar 4: Standardisasi XAI.
    """
    # 1. Grouping dan hitung rata-rata per kelas
    class_feature_means = df.groupby(target_col)[features].mean()
    
    # 2. Normalisasi Min-Max per kolom (fitur) agar bisa dibandingkan di heatmap
    scaler = MinMaxScaler()
    normalized_means = pd.DataFrame(
        scaler.fit_transform(class_feature_means),
        columns=class_feature_means.columns,
        index=class_feature_means.index
    )
    
    return normalized_means

def render_class_feature_heatmap(norm_df, title="Matriks Karakteristik Fitur Antar Kelas"):
    """
    Modul Visualisasi: Menampilkan heatmap relasi fitur vs kelas.
    """
    plt.figure(figsize=(14, 8))
    
    # Menggunakan colormap 'YlGnBu' atau 'magma' untuk kontras tinggi
    sns.heatmap(
        norm_df, 
        annot=True, 
        fmt=".2f", 
        cmap='YlGnBu', 
        linewidths=.5,
        cbar_kws={'label': 'Normalized Mean (0=Min, 1=Max)'}
    )
    
    plt.title(title, fontsize=15, fontweight='bold', pad=20)
    plt.xlabel("Daftar Fitur (Hybrid PCA + Clean)", fontsize=12)
    plt.ylabel("Kelas Rempah", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# --- EKSEKUSI ---
# Gunakan df_hybrid_final dan pastikan kolom 'class' ada di dalamnya
# Jika 'class' terpisah, gabungkan sementara untuk kalkulasi
df_temp = df_hybrid_final.copy()
df_temp['class'] = y_labels # y_labels adalah series target Anda

# Hitung matriks
class_matrix = calculate_class_feature_matrix(df_temp, df_hybrid_final.columns)

# Tampilkan Heatmap
render_class_feature_heatmap(class_matrix, title="Class-Feature Relationship Matrix\n(Karakteristik Visual Spesies Rempah)")

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

def render_class_proximity_dendrogram(df, target_col='class', title="Proksimitas Taksonomi Antar Kelas"):
    """
    Modul Visualisasi: Dendrogram untuk melihat kedekatan antar kelas rempah.
    Pilar 1: EDA Komprehensif.
    """
    # 1. Agregasi: Hitung rata-rata fitur untuk setiap kelas
    class_means = df.groupby(target_col).mean()
    
    # 2. Standardisasi Centroid (Penting agar jarak Euclidean adil)
    scaler = StandardScaler()
    class_means_scaled = scaler.fit_transform(class_means)
    
    # 3. Hitung Linkage (Metode Ward untuk klastering yang stabil)
    linkage_matrix = sch.linkage(class_means_scaled, method='ward')
    
    # 4. Plotting
    plt.figure(figsize=(10, 8))
    
    dendro = sch.dendrogram(
        linkage_matrix, 
        labels=class_means.index, 
        orientation='left', # Layout horizontal agar nama kelas mudah dibaca
        leaf_font_size=12,
        color_threshold=0.5 * max(linkage_matrix[:, 2])
    )
    
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.xlabel("Jarak Dissimilaritas (Euclidean)", fontsize=12)
    plt.ylabel("Spesies Rempah", fontsize=12)
    
    # Tambahkan grid untuk membantu pembacaan jarak
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

# --- EKSEKUSI ---
# Gunakan df_hybrid_final yang sudah digabung dengan y_labels
df_temp = df_hybrid_final.copy()
df_temp['class'] = y_labels 

render_class_proximity_dendrogram(df_temp, title="Dendrogram Proksimitas Spesies: Analisis Kemiripan Morfologi")

In [ ]:
df_temp.head()